In [7]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import warnings

warnings.filterwarnings('ignore')

In [9]:
def ucb_acquisition(x, gp, kappa = 2.0):
    #Upper Confidence Bound (UCB) acquisition function
    #x = x.reshape(1, -1)
    
    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    #ucb = mu + np.sqrt(beta) * sigma
    
    #return -ucb[0] # Return the negative UCB for minimisation
    return mu + kappa * sigma

def format_submission_string(x_best):
    x_formatted = [f"{coord:.6f}" for coord in x_best]
    return "-".join(x_formatted)

In [10]:
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy')
    
print(X)
print(Y)

[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837]


In [ ]:
X_init = np.array([
 [0.17152521, 0.34391687, 0.2487372 ],
 [0.24211446, 0.64407427, 0.27243281],
 [0.53490572, 0.39850092, 0.17338873],
 [0.49258141, 0.61159319, 0.34017639],
 [0.13462167, 0.21991724, 0.45820622],
 [0.34552327, 0.94135983, 0.26936348],
 [0.15183663, 0.43999062, 0.99088187],
 [0.64550284, 0.39714294, 0.91977134],
 [0.74691195, 0.28419631, 0.22629985],
 [0.17047699, 0.6970324 , 0.14916943],
 [0.22054934, 0.29782524, 0.34355534],
 [0.66601366, 0.67198515, 0.2462953 ],
 [0.04680895, 0.23136024, 0.77061759],
 [0.60009728, 0.72513573, 0.06608864],
 
 [0.96599485, 0.86111969, 0.56682913]]
)

y_init = np.array(
 [-0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758, -0.11062091,
 -0.39892551, -0.11386851, -0.13146061, -0.09418956, -0.04694741, -0.10596504,
 -0.11804826, -0.03637783, -0.05675837]
)

y_trans = y_init.copy()

kernel = Matern(length_scale=0.1, length_scale_bounds=(1e-1, 1e2), nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)

gp.fit(X_init, y_trans)

grid_size = 100
x1 = np.linspace(0, 1, grid_size)
x2 = np.linspace(0, 1, grid_size)
X_candidates = np.array([[i, j] for i in x1 for j in x2])

# --- 5. Evaluate acquisition function ---
acq_values = ucb_acquisition(X_candidates, gp, kappa=2.5)

# --- 6. Select next point ---
next_point = X_candidates[np.argmax(acq_values)]

print(format_submission_string(next_point))
#print(f"Next point to query:{next_point:.6f}")

ValueError: X has 2 features, but GaussianProcessRegressor is expecting 3 features as input.